In [ ]:
!git clone https://github.com/wlg1/seqcont_circ_expms.git
!git clone https://github.com/apartresearch/seqcont_circuits.git

In [ ]:
!nvidia-smi
import gc
import torch

try:
    del model
except NameError:
    pass
try:
    del pipe
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass

gc.collect()

torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()

if torch.cuda.is_available():
    print(torch.cuda.memory_summary())
!nvidia-smi

In [ ]:
!pip install transformers

In [ ]:
!pip install huggingface_hub
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))

In [ ]:
from __future__ import annotations
import os
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple, Iterable, Optional

import torch
import torch.nn as nn
import torch.distributed as dist
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
import pickle

# ========================= DDP helpers =========================

def ddp_is_on() -> bool:
    return dist.is_available() and dist.is_initialized()

def ddp_rank() -> int:
    return dist.get_rank() if ddp_is_on() else 0

def ddp_world() -> int:
    return dist.get_world_size() if ddp_is_on() else 1

def ddp_all_reduce_(t: torch.Tensor, op=dist.ReduceOp.SUM):
    if ddp_is_on():
        dist.all_reduce(t, op=op)

def ddp_barrier():
    if ddp_is_on():
        dist.barrier()

def setup_ddp_if_needed(enable: bool):
    if enable and not ddp_is_on():
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(int(os.environ.get("LOCAL_RANK", "0")))

# ========================= TL-compatible Dataset shim (robust) =========================

_CORRECT_ALIASES = [
    "correct", "answer", "ans", "target", "gold", "solution", "sol",
    "label_text", "corr"
]
_INCORRECT_ALIASES = [
    "incorrect", "foil", "distractor", "wrong", "negative", "alt", "other",
    "decoy", "incorr", "foils"
]
_CHOICES_ALIASES = ["choices", "options", "cands", "candidates"]
_LABELIDX_ALIASES = ["label", "label_idx", "answer_idx", "gold_idx", "target_idx"]

def _first_present_key(d: dict, keys: List[str]) -> Optional[str]:
    for k in keys:
        if k in d:
            return k
    return None

class Dataset:
    def __init__(
        self,
        examples: List[dict],
        tokenizer: AutoTokenizer,
        add_bos: bool = True,
        max_len: Optional[int] = None,
        require_ci: bool = True,
        correct_key: Optional[str] = None,
        incorrect_key: Optional[str] = None,
        choices_key: Optional[str] = None,
        label_idx_key: Optional[str] = None,
        incorrect_policy: str = "first", 
        append_space_to_prompt: bool = True,
    ):
        self.tokenizer = tokenizer
        self.add_bos = add_bos
        self.max_len = max_len
        self.require_ci = require_ci
        self.incorrect_policy = incorrect_policy
        self.append_space_to_prompt = append_space_to_prompt

        toks_list, attn_list = [], []
        corr_ids, inc_ids = [], []

        for idx, ex in enumerate(examples):
            raw_text = ex["text"]
            if self.append_space_to_prompt:
                if not raw_text or not raw_text[-1].isspace():
                    text = raw_text + " "
                else:
                    text = raw_text
            else:
                text = raw_text

            # ----- tokens/mask -----
            if add_bos and tokenizer.bos_token_id is not None:
                enc = tokenizer(text, add_special_tokens=False, truncation=True, max_length=max_len)
                ids = [tokenizer.bos_token_id] + enc["input_ids"]
            else:
                enc = tokenizer(text, add_special_tokens=True, truncation=True, max_length=max_len)
                ids = enc["input_ids"]
            toks = torch.tensor(ids, dtype=torch.long)
            attn = torch.ones_like(toks)
            toks_list.append(toks); attn_list.append(attn)

            # ----- correct/incorrect extraction -----
            got_ci = False
            corr_tok_id = None
            inc_tok_id = None

            def sanitize(s: str) -> str:
                # Remove leading whitespace so the continuation is the semantic token
                return s.lstrip()

            def ctx_first_id(s: str) -> int:
                return self._next_token_id_given_context(
                    text=text,
                    prefix_ids=ids,
                    s=sanitize(s),
                    tokenizer=tokenizer,
                    add_bos=add_bos,
                    max_len=max_len,
                )

            # 0) explicit int ids
            for tk in ["target_token_id", "correct_token_id", "gold_token_id"]:
                if tk in ex and isinstance(ex[tk], int):
                    corr_tok_id = ex[tk]; got_ci = True; break
            for fk in ["foil_token_id", "incorrect_token_id"]:
                if fk in ex and isinstance(ex[fk], int):
                    inc_tok_id = ex[fk]; got_ci = got_ci and True

            # 1) explicit CLI keys (strings)
            if not got_ci and (correct_key or incorrect_key):
                if correct_key and correct_key in ex and isinstance(ex[correct_key], str):
                    corr_tok_id = ctx_first_id(ex[correct_key]); got_ci = True
                if incorrect_key and incorrect_key in ex:
                    v = ex[incorrect_key]
                    if isinstance(v, str):
                        inc_tok_id = ctx_first_id(v); got_ci = got_ci and True
                    elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], str):
                        inc_tok_id = ctx_first_id(v[0]); got_ci = got_ci and True

            # 2) top-level aliases (includes corr/incorr)
            if not got_ci:
                ck = _first_present_key(ex, _CORRECT_ALIASES); fk = _first_present_key(ex, _INCORRECT_ALIASES)
                if ck and isinstance(ex[ck], str):
                    corr_tok_id = ctx_first_id(ex[ck])
                if fk:
                    v = ex[fk]
                    if isinstance(v, str):
                        inc_tok_id = ctx_first_id(v)
                    elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], str):
                        inc_tok_id = ctx_first_id(v[0])
                got_ci = (corr_tok_id is not None) and (inc_tok_id is not None)

            # 3) nested 'answers' dict
            if not got_ci and "answers" in ex and isinstance(ex["answers"], dict):
                ans = ex["answers"]
                ck = _first_present_key(ans, _CORRECT_ALIASES)
                fk = _first_present_key(ans, _INCORRECT_ALIASES)
                if ck and isinstance(ans[ck], str):
                    corr_tok_id = ctx_first_id(ans[ck])
                if fk:
                    v = ans[fk]
                    if isinstance(v, str):
                        inc_tok_id = ctx_first_id(v)
                    elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], str):
                        inc_tok_id = ctx_first_id(v[0])
                got_ci = (corr_tok_id is not None) and (inc_tok_id is not None)

            # 4) choices + label index
            if not got_ci:
                ck = choices_key or _first_present_key(ex, _CHOICES_ALIASES)
                lk = label_idx_key or _first_present_key(ex, _LABELIDX_ALIASES)
                if ck and lk and isinstance(ex[ck], list) and isinstance(ex[lk], int):
                    choices = ex[ck]
                    lab = ex[lk]
                    if 0 <= lab < len(choices) and isinstance(choices[lab], str):
                        corr_tok_id = ctx_first_id(choices[lab])
                        # pick incorrect
                        inc_idx = None
                        if "foils" in ex and isinstance(ex["foils"], list) and ex["foils"]:
                            inc_tok_id = ctx_first_id(ex["foils"][0])
                            inc_idx = -1
                        else:
                            if self.incorrect_policy == "first":
                                for j, c in enumerate(choices):
                                    if j != lab and isinstance(c, str):
                                        inc_idx = j; break
                            if inc_idx is not None and inc_idx >= 0:
                                inc_tok_id = ctx_first_id(choices[inc_idx])
                        got_ci = (corr_tok_id is not None) and (inc_tok_id is not None)

            if self.require_ci and not got_ci:
                sample_keys = list(ex.keys())
                raise ValueError(
                    "Each example must include a correct and incorrect string (or detectable schema).\n"
                    f"Example index: {idx}\nAvailable keys: {sample_keys}\n"
                    "Try passing --correct_key/--incorrect_key, or --choices_key/--label_idx_key."
                )

            if got_ci:
                corr_ids.append(corr_tok_id)
                inc_ids.append(inc_tok_id)

        maxT = max(x.numel() for x in toks_list)
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

        def left_pad_ids(x):
            return torch.cat([torch.full((maxT - x.numel(),), pad_id, dtype=torch.long), x])
        def left_pad_mask(x):
            return torch.cat([torch.zeros(maxT - x.numel(), dtype=torch.long), x])

        self.toks = torch.stack([left_pad_ids(x) for x in toks_list], dim=0)                # [N, T]
        self.attention_mask = torch.stack([left_pad_mask(x) for x in attn_list], dim=0)     # [N, T]
        self.pred_pos = self.attention_mask.sum(dim=1) - 1                                  # [N]

        # Label tensors (may be empty if require_ci=False)
        self.correct_token_ids = torch.tensor(corr_ids, dtype=torch.long) if corr_ids else torch.empty(0, dtype=torch.long)
        self.incorrect_token_ids = torch.tensor(inc_ids, dtype=torch.long) if inc_ids else torch.empty(0, dtype=torch.long)

    @staticmethod
    def _first_token_id(s: str, tokenizer: AutoTokenizer) -> int:
        ids_sp    = tokenizer(" " + s, add_special_tokens=False)["input_ids"]
        ids_plain = tokenizer(s, add_special_tokens=False)["input_ids"]
        if ids_sp and len(ids_sp) == 1: return ids_sp[0]
        if ids_plain and len(ids_plain) == 1: return ids_plain[0]
        if ids_sp: return ids_sp[0]
        if ids_plain: return ids_plain[0]
        ids_nl = tokenizer("\n" + s, add_special_tokens=False)["input_ids"]
        if ids_nl: return ids_nl[0]
        raise ValueError(f"String did not tokenize to any tokens: {repr(s)}")

    @staticmethod
    def _next_token_id_given_context(
        text: str,
        prefix_ids: List[int],
        s: str,
        tokenizer: AutoTokenizer,
        add_bos: bool,
        max_len: Optional[int],
    ) -> int:
        if add_bos and tokenizer.bos_token_id is not None:
            enc_full = tokenizer(text + s, add_special_tokens=False, truncation=True, max_length=max_len)
            full_ids = [tokenizer.bos_token_id] + enc_full["input_ids"]
            pref_len = len(prefix_ids)
        else:
            enc_pref = tokenizer(text, add_special_tokens=True, truncation=True, max_length=max_len)
            enc_full = tokenizer(text + s, add_special_tokens=True, truncation=True, max_length=max_len)
            full_ids = enc_full["input_ids"]
            pref_len = len(enc_pref["input_ids"])
        if len(full_ids) <= pref_len:
            return Dataset._first_token_id(s, tokenizer)
        return full_ids[pref_len]

# ========================= HF model wrapper (TL-like API) =========================

@dataclass
class ModelMeta:
    n_layers: int
    n_heads: int
    head_dim: int
    hidden_size: int
    arch: str  # "gpt2" or "llama"

class HFHookedLM(nn.Module):
    def __init__(self, model_name: str, dtype: Optional[torch.dtype] = None, device: Optional[str] = None):
        super().__init__()
        self.model_name = model_name
        self.config = AutoConfig.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if dtype is None:
            dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        self.attn_impl = "eager"
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            attn_implementation=self.attn_impl,
        )
        self.model.eval()

        if device is None:
            device = f"cuda:{int(os.environ.get('LOCAL_RANK', 0))}" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)
        self.to(self.device)

        self.meta = self._introspect_arch()

        self._perm_hooks: List[torch.utils.hooks.RemovableHandle] = []
        self._temp_hooks: List[torch.utils.hooks.RemovableHandle] = []

        self._head_means: Optional[Dict[Tuple[int, int], torch.Tensor]] = None
        self._mlp_means: Optional[Dict[int, torch.Tensor]] = None

    # ---- Arch utils ----
    def _introspect_arch(self) -> ModelMeta:
        cfg = self.model.config
        if hasattr(cfg, "n_layer"):  # GPT-2
            L = cfg.n_layer; H = cfg.n_head; d = cfg.n_embd; hd = d // H; arch = "gpt2"
        else:  # LLaMA-style
            L = cfg.num_hidden_layers; H = cfg.num_attention_heads; d = cfg.hidden_size; hd = d // H; arch = "llama"
        return ModelMeta(L, H, hd, d, arch)

    def _block(self, layer: int) -> nn.Module:
        return self.model.transformer.h[layer] if self.meta.arch == "gpt2" else self.model.model.layers[layer]

    def _o_proj(self, layer: int) -> nn.Module:
        return self._block(layer).attn.c_proj if self.meta.arch == "gpt2" else self._block(layer).self_attn.o_proj

    def _mlp(self, layer: int) -> nn.Module:
        return self._block(layer).mlp

    # ---- TL-like API ----
    def reset_hooks(self, including_permanent: bool = False):
        for h in self._temp_hooks:
            h.remove()
        self._temp_hooks = []
        if including_permanent:
            for h in self._perm_hooks:
                h.remove()
            self._perm_hooks = []

    @torch.no_grad()
    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None):
        input_ids = input_ids.to(self.device)
        kwargs = {}
        if attention_mask is not None:
            kwargs["attention_mask"] = attention_mask.to(self.device)
        return self.model(input_ids=input_ids, **kwargs).logits

    @torch.no_grad()
    def run_with_cache(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None):
        """
        Return (logits, cache) where cache contains:
          ("attn", L):   [B, H, T, T] attention probabilities (eager only)
          ("attn_z", L): [B, T, H*D]  concatenated per-head z before W_O
          ("mlp_out", L):[B, T, d]    MLP output
        """
        cache: Dict[Tuple[str, int], List[torch.Tensor]] = {}
        H, D = self.meta.n_heads, self.meta.head_dim

        # capture z at pre-W_O
        for L in range(self.meta.n_layers):
            def _pre(layer=L):
                def hook(m, inp):
                    x = inp[0]  # [B, T, H*D]
                    cache.setdefault(("attn_z", layer), []).append(x.detach())
                return hook
            self._temp_hooks.append(self._o_proj(L).register_forward_pre_hook(_pre()))

        # capture MLP output
        for L in range(self.meta.n_layers):
            def _mlp_hook(layer=L):
                def hook(m, inp, out):
                    y = out[0] if isinstance(out, tuple) else out
                    cache.setdefault(("mlp_out", layer), []).append(y.detach())
                return hook
            self._temp_hooks.append(self._mlp(L).register_forward_hook(_mlp_hook()))

        # forward (ask for attentions in eager)
        input_ids = input_ids.to(self.device)
        kwargs = {"output_attentions": True}
        if attention_mask is not None:
            kwargs["attention_mask"] = attention_mask.to(self.device)
        out = self.model(input_ids=input_ids, **kwargs)
        logits = out.logits

        # attentions
        if getattr(out, "attentions", None) is not None:
            for L, att in enumerate(out.attentions):
                cache[("attn", L)] = [att.detach()]

        # merge lists
        flat: Dict[Tuple[str, int], torch.Tensor] = {}
        for k, lst in cache.items():
            flat[k] = torch.cat(lst, dim=0) if isinstance(lst, list) else lst

        self.reset_hooks(including_permanent=False)
        return logits, flat

    # ---- Means (from Dataset 2) & ablation hooks ----
    @torch.no_grad()
    def _compute_means_from_dataset2(self, dataset_2: Dataset):
        H, D, d_model = self.meta.n_heads, self.meta.head_dim, self.meta.hidden_size
    
        z_sum   = {L: torch.zeros(H, D, device=self.device, dtype=self.model.dtype) for L in range(self.meta.n_layers)}
        mlp_sum = {L: torch.zeros(d_model, device=self.device, dtype=self.model.dtype) for L in range(self.meta.n_layers)}
    
        # Run once to cache all layer activations
        _, cache = self.run_with_cache(dataset_2.toks, dataset_2.attention_mask)
    
        # Build mask [N, T, 1] on the correct device/dtype so MLP/head outputs at pad positions are EXCLUDED
        mask = dataset_2.attention_mask.to(self.device).to(self.model.dtype).unsqueeze(-1)  # [N, T, 1]
        valid_count = mask.sum()  # scalar in same dtype; counts real tokens
    
        # Sum with masking
        for L in range(self.meta.n_layers):
            # Heads: pre-W_O states ("attn_z"): [N, T, H*D] -> [N, T, H, D]
            Z = cache.get(("attn_z", L), None)
            if Z is not None:
                Z = Z.view(Z.shape[0], Z.shape[1], H, D)
                Z_masked = Z * mask.unsqueeze(2)  # [N, T, 1, 1] broadcast
                z_sum[L] += Z_masked.sum(dim=(0, 1))  # -> [H, D]
    
            # MLP output: [N, T, d_model]
            M = cache.get(("mlp_out", L), None)
            if M is not None:
                M_masked = M * mask  # [N, T, d_model]
                mlp_sum[L] += M_masked.sum(dim=(0, 1))  # -> [d_model]
    
        # DDP aggregate
        for L in range(self.meta.n_layers):
            ddp_all_reduce_(z_sum[L]); ddp_all_reduce_(mlp_sum[L])
        ddp_all_reduce_(valid_count)
    
        denom = max(1.0, float(valid_count.item()))
    
        head_means = {}
        for L in range(self.meta.n_layers):
            mean_layer = z_sum[L] / denom  # [H, D]
            for h in range(H):
                head_means[(L, h)] = mean_layer[h].detach().clone()
    
        mlp_means = {L: (mlp_sum[L] / denom).detach().clone() for L in range(self.meta.n_layers)}
    
        self._head_means = head_means
        self._mlp_means  = mlp_means



    def _ensure_means(self, dataset_2: Dataset):
        if self._head_means is None or self._mlp_means is None:
            self._compute_means_from_dataset2(dataset_2)

    def add_mean_ablation_hooks(self, heads_keep: Iterable[Tuple[int,int]], mlps_keep: Iterable[int], dataset_2: Dataset):
        """
        Permanent mean-ablation hooks (removed by reset_hooks(including_permanent=True)):
          - Heads NOT in heads_keep are mean-ablated at pre-W_O.
          - MLPs NOT in mlps_keep are mean-ablated at MLP output.
        """
        self._ensure_means(dataset_2)
        heads_keep = set(heads_keep); mlps_keep = set(mlps_keep)
        H, D = self.meta.n_heads, self.meta.head_dim

        # Heads (pre-W_O)
        for L in range(self.meta.n_layers):
            abl_heads = [h for h in range(H) if (L, h) not in heads_keep]
            if not abl_heads:
                continue
            means = torch.stack([self._head_means[(L, h)].to(self.device) for h in abl_heads], dim=0)
            def _pre(layer=L, abl_h=abl_heads, means_local=means):
                def hook(m, inp):
                    x = inp[0]  # [B, T, H*D]
                    B, T, _ = x.shape
                    xv = x.view(B, T, H, D).clone()
                    for j, h in enumerate(abl_h):
                        xv[:, :, h, :] = means_local[j].view(1, 1, D)
                    return (xv.view(B, T, H*D),)
                return hook
            self._perm_hooks.append(self._o_proj(L).register_forward_pre_hook(_pre()))

        # MLPs (output)
        for L in range(self.meta.n_layers):
            if L in mlps_keep:
                continue
            mean_vec = self._mlp_means[L].to(self.device)
            def _mlp(mean=mean_vec):
                def hook(m, inp, out):
                    y = out[0] if isinstance(out, tuple) else out
                    B, T, _ = y.shape
                    return mean.view(1, 1, -1).expand(B, T, -1)
                return hook
            self._perm_hooks.append(self._mlp(L).register_forward_hook(_mlp()))

# ========================= Metric: correct - incorrect =========================

@torch.no_grad()
def get_logit_diff(logits: torch.Tensor, dataset: Dataset) -> float:
    """
    mean over examples of logits[i, pos_i, correct_id] - logits[i, pos_i, incorrect_id]
    """
    logits = logits.float()
    if dataset.correct_token_ids.numel() == 0 or dataset.incorrect_token_ids.numel() == 0:
        raise ValueError("get_logit_diff called but dataset lacks correct/incorrect ids. Pass require_ci=True for that dataset.")

    B = logits.shape[0]
    device = logits.device
    pos = dataset.pred_pos.to(device)  # [B]
    corr = dataset.correct_token_ids.to(device).view(-1, 1)  # [B,1]
    inc  = dataset.incorrect_token_ids.to(device).view(-1, 1)  # [B,1]

    logits_at_pos = logits[torch.arange(B, device=device), pos]  # [B, V]

    corr_logit = logits_at_pos.gather(1, corr).squeeze(1)  # [B]
    inc_logit  = logits_at_pos.gather(1, inc).squeeze(1)   # [B]
    return (corr_logit - inc_logit).mean().item()


def add_ablation_hook_MLP_head(model: HFHookedLM, dataset_2: Dataset,
                               heads_not_ablate: Iterable[Tuple[int,int]], mlps_not_ablate: Iterable[int]) -> HFHookedLM:
    """Permanent mean-ablation for all nodes NOT in the candidate circuit."""
    model.add_mean_ablation_hooks(heads_keep=heads_not_ablate, mlps_keep=mlps_not_ablate, dataset_2=dataset_2)
    return model

@torch.no_grad()
def ablate_head_from_full(heads_keep: Iterable[Tuple[int,int]], model: HFHookedLM,
                          dataset_1: Dataset, dataset_2: Dataset, orig_score: float, print_output: bool = True) -> torch.Tensor:
    """Keep provided heads; mean-ablade all other heads. DO NOT ablate any MLPs."""
    model.reset_hooks(including_permanent=True)
    model.add_mean_ablation_hooks(heads_keep=heads_keep, mlps_keep=list(range(model.meta.n_layers)), dataset_2=dataset_2)
    logits = model(dataset_1.toks, dataset_1.attention_mask)
    new_score = get_logit_diff(logits, dataset_1)
    perc = 100.0 * new_score / max(1e-9, orig_score)
    if print_output and ddp_rank() == 0:
        print(f"(cand circuit / full) %: {perc:.4f}")
    return torch.tensor(perc)

@torch.no_grad()
def ablate_MLP_from_full(mlps_keep: Iterable[int], model: HFHookedLM,
                         dataset_1: Dataset, dataset_2: Dataset, orig_score: float, print_output: bool = True) -> torch.Tensor:
    """Keep provided MLPs; mean-ablade all other MLPs. DO NOT ablate any heads."""
    model.reset_hooks(including_permanent=True)
    all_heads = [(L, h) for L in range(model.meta.n_layers) for h in range(model.meta.n_heads)]
    model.add_mean_ablation_hooks(heads_keep=all_heads, mlps_keep=mlps_keep, dataset_2=dataset_2)
    logits = model(dataset_1.toks, dataset_1.attention_mask)
    new_score = get_logit_diff(logits, dataset_1)
    perc = 100.0 * new_score / max(1e-9, orig_score)
    if print_output and ddp_rank() == 0:
        print(f"(cand circuit / full) %: {perc:.4f}")
    return torch.tensor(perc)

# ========================= Main =========================

def main():
    import shlex
    import argparse

    def build_parser():
        ap = argparse.ArgumentParser()
        ap.add_argument("--model", type=str, default="gpt2")
        ap.add_argument("--data_task_dir", type=str, required=True)
        ap.add_argument("--data_task_rand", type=str, required=True)
        ap.add_argument("--ddp", action="store_true")
        ap.add_argument("--add_bos", action="store_true", default=True)
        ap.add_argument("--max_len", type=int, default=None)
        ap.add_argument("--correct_key", type=str, default=None)
        ap.add_argument("--incorrect_key", type=str, default=None)
        ap.add_argument("--choices_key", type=str, default=None)
        ap.add_argument("--label_idx_key", type=str, default=None)
        ap.add_argument("--incorrect_policy", type=str, default="first", choices=["first"])
        ap.add_argument("--append_space", dest="append_space", action="store_true", default=True,
                    help="Ensure one trailing space at end of every prompt (default: on)")
        ap.add_argument("--no-append_space", dest="append_space", action="store_false")
        return ap

    parser = build_parser()
    arg_str = """
    --data_task_dir seqcont_circuits/data/numerals/numerals_prompts_done.pkl
    --data_task_rand seqcont_circuits/data/numerals/randDS_numerals.pkl
    --model meta-llama/Llama-3.1-8B
    --max_len 256
    --correct_key corr
    --incorrect_key incorr
    """
    args = parser.parse_args(shlex.split(arg_str))

    setup_ddp_if_needed(args.ddp)

    # Load model
    lm = HFHookedLM(args.model)
    tok = lm.tokenizer

    # Load data
    with open(args.data_task_dir, "rb") as f:
        ds1_list = pickle.load(f)
    with open(args.data_task_rand, "rb") as f:
        ds2_list = pickle.load(f)

    # Dataset 1: requires correct/incorrect (corr/incorr auto-detected or via overrides)
    dataset_1 = Dataset(
        ds1_list, tok,
        add_bos=args.add_bos, max_len=args.max_len, require_ci=True,
        correct_key=args.correct_key, incorrect_key=args.incorrect_key,
        choices_key=args.choices_key, label_idx_key=args.label_idx_key,
        incorrect_policy=args.incorrect_policy,
        append_space_to_prompt=args.append_space,
    )

    # Dataset 2: labels optional (we just need tokens for means)
    dataset_2 = Dataset(
        ds2_list, tok,
        add_bos=args.add_bos, max_len=args.max_len, require_ci=False,
        correct_key=None, incorrect_key=None,
        choices_key=None, label_idx_key=None,
        append_space_to_prompt=args.append_space,
    )

    # Sanity: how many examples have identical IDs?
    if ddp_rank() == 0:
        same = (dataset_1.correct_token_ids == dataset_1.incorrect_token_ids).sum().item()
        print(f"Examples where corr_id == inc_id: {same} / {len(dataset_1.correct_token_ids)}")

    # Original score
    lm.reset_hooks(including_permanent=True)
    logits_original = lm(dataset_1.toks, dataset_1.attention_mask)
    orig_score = get_logit_diff(logits_original, dataset_1)
    if ddp_rank() == 0:
        print(f"Original score (correct - incorrect): {orig_score:.6f}")

    @torch.no_grad()
    def get_top1_accuracy(logits: torch.Tensor, dataset: Dataset) -> float:
        logits = logits.float()
        if dataset.correct_token_ids.numel() == 0:
            raise ValueError("get_top1_accuracy called but dataset lacks correct ids. Pass require_ci=True for that dataset.")
        B = logits.shape[0]
        device = logits.device
        pos = dataset.pred_pos.to(device)                           # [B]
        corr = dataset.correct_token_ids.to(device)                 # [B]
        logits_at_pos = logits[torch.arange(B, device=device), pos] # [B, V]
        pred = logits_at_pos.argmax(dim=-1)                         # [B]
        return (pred == corr).float().mean().item()
        
    orig_acc = get_top1_accuracy(logits_original, dataset_1)
    print(f"Original top-1 accuracy: {orig_acc*100:.2f}%")

    # Candidate circuit % check (no heads/MLPs kept)
    heads_not_ablate: List[Tuple[int,int]] = []
    mlps_not_ablate: List[int] = []
    lm.reset_hooks(including_permanent=True)
    add_ablation_hook_MLP_head(lm, dataset_2, heads_not_ablate, mlps_not_ablate)
    new_logits = lm(dataset_1.toks, dataset_1.attention_mask)
    new_score = get_logit_diff(new_logits, dataset_1)
    circ_score = (100.0 * new_score / max(1e-9, orig_score))
    if ddp_rank() == 0:
        print(f"(cand circuit / full) %: {circ_score:.4f}")
    del new_logits

    print("================================")

    # Head scan (drop-one from full)
    L, H = lm.meta.n_layers, lm.meta.n_heads
    circ = [(layer, head) for layer in range(L) for head in range(H)]
    to_loop = [(layer, head) for layer in range(L) for head in range(H)]
    if ddp_rank() == 0:
        print("Starting head drop-one from FULL...")
    lh_scores: Dict[Tuple[int,int], float] = {}
    my_list = to_loop[ddp_rank()::max(1, ddp_world())]
    for lh in my_list:
        print(lh)
        copy_circuit = circ.copy(); copy_circuit.remove(lh)
        perc = ablate_head_from_full(copy_circuit, lm, dataset_1, dataset_2, orig_score, print_output=False).item()
        lh_scores[lh] = float(perc)
    if ddp_is_on():
        obj_list = [None for _ in range(ddp_world())]
        dist.all_gather_object(obj_list, lh_scores)
        merged = {}
        for d in obj_list:
            merged.update(d)
        lh_scores = merged
    if ddp_rank() == 0:
        with open("lh_scores.json", "w") as f:
            json.dump({str(k): v for k, v in sorted(lh_scores.items())}, f, indent=2)
        print("Saved head drop-one results to lh_scores.json")

    print("================================")

    # MLP scan (drop-one from full)
    if ddp_rank() == 0:
        print("Starting MLP drop-one from FULL...")
    for i in range(L):
        keep = [l for l in range(L) if l != i]
        perc = ablate_MLP_from_full(keep, lm, dataset_1, dataset_2, orig_score, print_output=False).item()
        if ddp_rank() == 0:
            print(i, perc)

    if ddp_rank() == 0:
        print("Done.")



In [ ]:
import sys

sys.argv = [
    "hf_seq_circuit_runpod.py",
    "--model", "meta-llama/Llama-3.1-8B", #nvidia/Llama-3.1-Minitron-4B-Depth-Base
    "--data_task_dir", "seqcont_circuits/data/numerals/numerals_prompts_done.pkl",
    "--data_task_rand", "seqcont_circuits/data/numerals/randDS_numerals.pkl",
    "--max_len", "256",
    "--correct_key", "corr",
    "--incorrect_key", "incorr",
    # Don't pass --ddp inside a single notebook process.
]

# call the script’s entrypoint
main()
